<a href="https://colab.research.google.com/github/rickyajs/Data-Science-2026/blob/main/Pertemuan_12_RickyArmandaJayaSirait_240401020219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nama  : Ricky Armanda Jaya Sirait
# Nim   : 240401020219
#Kelas  : IF401
#Prodi  : PJJ S1 Informatika

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("=== LANGKAH 1: GENERATE & EKSPLORASI DATASET ===")
np.random.seed(42) # [cite: 179]
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'] # [cite: 180, 181]

# Buat 50 transaksi sintetis (tiap transaksi berisi 2-5 produk)
transaksi = [] # [cite: 183]
for _ in range(50): # [cite: 184]
    n_item = np.random.randint(2, 6) # [cite: 185]
    transaksi.append(list(np.random.choice(produk, n_item, replace=False))) # [cite: 186]

# Suntikkan pola tersembunyi: Roti sering dibeli bersama Selai
for i in range(0, 20): # [cite: 188]
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]: # [cite: 189]
        transaksi[i].append('Selai') # [cite: 189]

print('Contoh transaksi:', transaksi[:3]) # [cite: 190]
print('Jumlah transaksi:', len(transaksi)) # [cite: 191]
print("-" * 50)

print("\n=== LANGKAH 2: ONE-HOT ENCODING TRANSAKSI ===")
te = TransactionEncoder() # [cite: 195]
te_ary = te.fit(transaksi).transform(transaksi) # [cite: 197]
df = pd.DataFrame(te_ary, columns=te.columns_) # [cite: 198]

print("Lima baris pertama data One-Hot Encoded:")
print(df.head())
print("-" * 50)

print("\n=== LANGKAH 3: CARI FREQUENT ITEMSET ===")
# Eksperimen dengan beberapa nilai min_support
for ms in [0.05, 0.1, 0.2]: # [cite: 201]
    freq = apriori(df, min_support=ms, use_colnames=True) # [cite: 201]
    print(f'min_support={ms}: {len(freq)} itemset ditemukan') # [cite: 201]

# Gunakan min_support yang menghasilkan jumlah itemset wajar (0.1)
freq_items = apriori(df, min_support=0.1, use_colnames=True) # [cite: 201]
freq_items = freq_items.sort_values('support', ascending=False) # [cite: 201]

print("\nTop 10 Frequent Itemset:")
print(freq_items.head(10)) # [cite: 201]
print("-" * 50)

print("\n=== LANGKAH 4: BENTUK & SARING ATURAN ASOSIASI ===")
rules = association_rules(freq_items, metric='confidence', min_threshold=0.5) # [cite: 203]
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False) # [cite: 203]

print("Top 10 Aturan Asosiasi Terkuat (berdasarkan Lift):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)) # [cite: 203]
print("-" * 50)

print("\n=== LANGKAH 5: CONTENT-BASED FILTERING KATALOG PRODUK ===")
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy', 'Dairy','Minuman','Bumbu','Minuman','Dairy'] # [cite: 206]
})

fitur = pd.get_dummies(katalog['kategori']) # [cite: 206]
sim_matrix = cosine_similarity(fitur) # [cite: 206]

def rekomendasi_serupa(nama_produk, top_n=3): # [cite: 207]
    idx = katalog.index[katalog['produk'] == nama_produk][0] # [cite: 207]
    skor = list(enumerate(sim_matrix[idx])) # [cite: 207]
    skor = sorted(skor, key=lambda x: x[1], reverse=True) # [cite: 207]
    skor = [s for s in skor if s[0] != idx][:top_n] # [cite: 207]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist() # [cite: 207]

print('Mirip dengan Roti berdasarkan kategori:', rekomendasi_serupa('Roti')) # [cite: 207]
print("-" * 50)

print("\n=== LANGKAH 6: PERBANDINGAN KEDUA PENDEKATAN ===")
produk_target = 'Roti' # [cite: 208]

# Cari dari association rules
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)] # [cite: 208]

print('Rekomendasi dari Association Rules untuk "Roti":')
print(rules_terkait[['consequents', 'lift']].head()) # [cite: 208]

print('\nRekomendasi dari Content-Based untuk "Roti":')
print(rekomendasi_serupa(produk_target)) # [cite: 208]
print("-" * 50)

=== LANGKAH 1: GENERATE & EKSPLORASI DATASET ===
Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50
--------------------------------------------------

=== LANGKAH 2: ONE-HOT ENCODING TRANSAKSI ===
Lima baris pertama data One-Hot Encoded:
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False
--------------------------------------------------

=== LANGKAH 3: CARI FREQUENT ITEMSET ===
min_s